# 14 · Full Pipeline Batch Run — LangGraph RAG + Knowledge Graph + Prompt Engineering

**Memorial Sloan Kettering | Goel Lab**

Runs the full LangGraph RAG extraction pipeline described in `README.md §LangGraph RAG Extraction Pipeline` on all 452 OCR-cached source documents, builds a NetworkX knowledge graph, and benchmarks prompt engineering techniques from the prompt library.

---

### Pipeline Stages

| Stage | Description | Output |
|-------|-------------|--------|
| **1 — Discover** | Scan OCR cache, build case list | `cases` list |
| **2 — Batch RAG** | LangGraph graph per case (load → index → retrieve → extract → verify → adjudicate) | `experiments/runs/{run_id}/` |
| **3 — Knowledge Graph** | Aggregate results → NetworkX DiGraph → GraphML | `data/knowledge_graph/{run_id}_kg.graphml` |
| **4 — Prompt Comparison** | Direct Claude calls with each prompt library technique on a representative sample | `experiments/prompt_comparison/` |
| **5 — Summary** | Fabrication / omission / correct rates per feature, per domain, per prompt technique | `reports/` |

In [ ]:
import json, os, sys, time, warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv(override=True)
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# ── Repo root on sys.path so src.* imports resolve ────────────────────────────
REPO_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center"
        r"\Documents\GitHub\llm_summarization_br_ca",
    )
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.workflows.orchestration import run_batch, run_single_case
from src.graph.build_graph import build_networkx_graph, results_to_kg, save_graphml
from src.rag.feature_queries import FEATURE_LIST, FEATURES
from src.utils.io_utils import generate_run_id

import anthropic

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
assert api_key.startswith("sk-ant"), "ANTHROPIC_API_KEY not set — check .env"
CLIENT = anthropic.Anthropic(api_key=api_key)

print(f"Repo root : {REPO_ROOT}")
print(f"Features  : {len(FEATURE_LIST)}")
print("Imports OK")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
OCR_CACHE       = Path(r"C:\Users\jamesr4\loc\data_private\ocr_cache")
EXPERIMENTS_DIR = REPO_ROOT / "experiments"
KG_DIR          = REPO_ROOT / "data" / "knowledge_graph"
PROMPT_LIB      = REPO_ROOT / "prompts" / "library"
REPORTS_DIR     = REPO_ROOT / "reports"
PROMPT_CMP_DIR  = EXPERIMENTS_DIR / "prompt_comparison"

for d in (KG_DIR, REPORTS_DIR, PROMPT_CMP_DIR):
    d.mkdir(parents=True, exist_ok=True)

MODEL           = "claude-sonnet-4-6"
MAX_TOKENS      = 2048         # for direct prompt-comparison calls
BATCH_RUN_ID    = os.getenv("BATCH_RUN_ID") or generate_run_id()   # persist across reruns
N_CASES_LIMIT   = None         # set to an int (e.g. 20) to test on a subset; None = all 452
N_PROMPT_CMP    = 10           # cases used for prompt-engineering comparison

print(f"OCR cache     : {OCR_CACHE}")
print(f"Batch run_id  : {BATCH_RUN_ID}")
print(f"Cases limit   : {N_CASES_LIMIT or 'all'}")
print(f"Prompt sample : {N_PROMPT_CMP} cases × each technique")

---
## Section 1 · Discover OCR Text Files

In [ ]:
# ── Enumerate all cached OCR text files ───────────────────────────────────────
txt_files = sorted(OCR_CACHE.glob("CASE_*.txt"))
if N_CASES_LIMIT:
    txt_files = txt_files[:N_CASES_LIMIT]

cases = []
char_counts = []
for f in txt_files:
    text = f.read_text(encoding="utf-8", errors="replace")
    cases.append({"case_id": f.stem, "ocr_text": text})
    char_counts.append(len(text))

char_counts = np.array(char_counts)
print(f"Cases found   : {len(cases)}")
print(f"Text length   : median={np.median(char_counts):,.0f} "
      f"min={char_counts.min():,} max={char_counts.max():,} chars")

# Size distribution
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(char_counts / 1000, bins=40, color="steelblue", edgecolor="white")
ax.set_xlabel("OCR text length (thousand chars)")
ax.set_ylabel("Number of cases")
ax.set_title(f"Source document size distribution  (n={len(cases)})", fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "14_case_text_length_distribution.png", dpi=150)
plt.show()

---
## Section 2 · Run LangGraph RAG Pipeline (Full Batch)

The graph topology is:
```
load_case → index_case → next_feature → retrieve → extract → verify
  verify → adjudicate | rewrite_query | self_consistency
  rewrite_query → retrieve_again → extract → verify
  self_consistency → adjudicate
  adjudicate → next_feature | aggregate → END
```

Results are saved incrementally to `experiments/runs/{BATCH_RUN_ID}/` so the cell is **resume-safe** — already-processed cases are skipped.

In [ ]:
# ── Resume helper — find already-completed cases ──────────────────────────────
def get_completed_cases(run_id: str) -> set:
    run_dir = EXPERIMENTS_DIR / "runs" / run_id
    if not run_dir.exists():
        return set()
    return {p.stem for p in run_dir.glob("*.json") if p.stem != "run_manifest"}

completed = get_completed_cases(BATCH_RUN_ID)
pending   = [c for c in cases if c["case_id"] not in completed]

print(f"Total cases  : {len(cases)}")
print(f"Already done : {len(completed)}")
print(f"Pending      : {len(pending)}")

In [ ]:
# ── Run batch ─────────────────────────────────────────────────────────────────
# This cell processes all pending cases.  Re-run at any time to resume.
# Each case takes ~30–60 s depending on document length.
# Estimated time: ~4–8 hours for all 452 cases.

if not pending:
    print("All cases already processed — loading results from disk.")
else:
    print(f"Starting batch: {len(pending)} cases, run_id={BATCH_RUN_ID}")
    t0 = time.time()
    batch_results = run_batch(
        cases=pending,
        prompt_id="rag_verify_v1",
        model_id=MODEL,
        run_id=BATCH_RUN_ID,
        save_results=True,
    )
    elapsed = time.time() - t0
    n_ok  = sum(1 for r in batch_results if "features" in r)
    n_err = sum(1 for r in batch_results if "error" in r)
    print(f"\nBatch complete: {n_ok} OK, {n_err} errors  ({elapsed/60:.1f} min)")

In [ ]:
# ── Load all saved results from disk ─────────────────────────────────────────
run_dir = EXPERIMENTS_DIR / "runs" / BATCH_RUN_ID
all_results = []
error_cases = []

for p in sorted(run_dir.glob("*.json")):
    if p.stem == "run_manifest":
        continue
    try:
        r = json.loads(p.read_text(encoding="utf-8"))
        if "features" in r:
            all_results.append(r)
        else:
            error_cases.append(r)
    except Exception as e:
        print(f"[WARN] could not read {p.name}: {e}")

print(f"Loaded {len(all_results)} successful results, {len(error_cases)} errors")

# Build flat feature table
rows = []
for r in all_results:
    for feat, fd in r.get("features", {}).items():
        rows.append({
            "case_id":    r["case_id"],
            "run_id":     r["run_id"],
            "prompt_id":  r.get("prompt_id", ""),
            "feature":    feat,
            "display":    FEATURES.get(feat, {}).get("display_name", feat),
            "value":      fd.get("value", ""),
            "confidence": fd.get("confidence", 0.0),
            "supported":  fd.get("supported"),
            "verdict":    fd.get("verdict", ""),
            "verification_confidence": fd.get("verification_confidence"),
        })

df_features = pd.DataFrame(rows)
print(df_features.shape)
df_features.head(3)

---
## Section 3 · Knowledge Graph

Converts batch results into a typed `KnowledgeGraph` (Patient → Feature → Claim → Verdict nodes) and exports to GraphML for downstream analysis or Neo4j loading.

In [ ]:
# ── Build KnowledgeGraph + NetworkX DiGraph ───────────────────────────────────
kg   = results_to_kg(all_results)
G    = build_networkx_graph(kg)

graphml_path = KG_DIR / f"{BATCH_RUN_ID}_kg.graphml"
save_graphml(G, graphml_path)

# Node and edge summary
node_types = {}
for n, d in G.nodes(data=True):
    t = d.get("type", "Unknown")
    node_types[t] = node_types.get(t, 0) + 1

edge_types = {}
for u, v, d in G.edges(data=True):
    r = d.get("relation", "Unknown")
    edge_types[r] = edge_types.get(r, 0) + 1

print(f"Nodes : {G.number_of_nodes():,}")
for t, cnt in sorted(node_types.items()):
    print(f"  {t:<30} {cnt:>6,}")
print(f"Edges : {G.number_of_edges():,}")
for r, cnt in sorted(edge_types.items()):
    print(f"  {r:<30} {cnt:>6,}")
print(f"\nGraphML saved → {graphml_path}")

In [ ]:
# ── Node type distribution bar chart ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

pd.Series(node_types).sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Node types", fontsize=12)
axes[0].set_xlabel("Count")

pd.Series(edge_types).sort_values().plot.barh(ax=axes[1], color="darkorange")
axes[1].set_title("Edge (relation) types", fontsize=12)
axes[1].set_xlabel("Count")

plt.suptitle("Knowledge Graph Composition", fontsize=13)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "14_kg_node_edge_types.png", dpi=150)
plt.show()

In [ ]:
# ── Visualise a subgraph: one patient + all feature/verdict nodes ─────────────
import random
random.seed(42)

patient_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "Patient"]
if patient_nodes:
    sample_patient = random.choice(patient_nodes[:20])   # pick from first 20
    neighbors_1 = set(G.successors(sample_patient))
    neighbors_2 = set()
    for n in neighbors_1:
        neighbors_2.update(G.successors(n))

    sub_nodes = {sample_patient} | neighbors_1 | neighbors_2
    SG = G.subgraph(sub_nodes)

    color_map = {
        "Patient":           "#e74c3c",
        "ClinicalFeature":   "#3498db",
        "ExtractionClaim":   "#2ecc71",
        "ValidationVerdict": "#f39c12",
        "SourceDocument":    "#9b59b6",
        "EvidenceChunk":     "#95a5a6",
    }
    colors  = [color_map.get(SG.nodes[n].get("type", ""), "#bdc3c7") for n in SG.nodes]
    labels  = {n: SG.nodes[n].get("feature_name", "")[:15] or n[:10] for n in SG.nodes}
    pos     = nx.spring_layout(SG, seed=42, k=1.5)

    fig, ax = plt.subplots(figsize=(14, 9))
    nx.draw_networkx(SG, pos=pos, ax=ax, node_color=colors, node_size=300,
                     labels=labels, font_size=7, edge_color="#ccc", arrows=True,
                     arrowsize=10)

    from matplotlib.patches import Patch
    legend = [Patch(color=c, label=t) for t, c in color_map.items()]
    ax.legend(handles=legend, loc="upper left", fontsize=8, framealpha=0.7)
    ax.set_title(f"Knowledge graph — case {sample_patient}  ({len(SG)} nodes)",
                 fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "14_kg_sample_subgraph.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## Section 4 · Prompt Engineering Comparison

Runs `N_PROMPT_CMP` cases through each prompt technique in the library using a direct Claude API call (single-pass extraction), then compares correct / fabrication / omission rates across techniques.

**Techniques benchmarked** (from `prompts/library/prompt_txt_files_manifest.csv`):

| Category | Files |
|----------|-------|
| Baseline structured | baseline_structured_prompt_initial |
| Zero-shot | zero_shot_1, zero_shot_2, zero_shot_structured_extraction_prod |
| Chain-of-thought | chain_of_thought_1, chain_of_thought_2, chain_of_thought_3 |
| Few-shot | few_shot_1, few_shot_2 |
| RAG template | rag_1, rag_2 |
| ReAct | react_1, react_2 |
| Program-aided | program_aided_1, program_aided_2 |

In [ ]:
# ── Load prompt library ───────────────────────────────────────────────────────
manifest = pd.read_csv(PROMPT_LIB / "prompt_txt_files_manifest.csv")
print(manifest.to_string(index=False))

# Load all prompt texts
prompts = {}
for _, row in manifest.iterrows():
    p = PROMPT_LIB / row["TXT file"]
    if p.exists():
        prompts[row["Column name"]] = {
            "technique":  row["Prompting technique"],
            "name":       row["Column name"],
            "text":       p.read_text(encoding="utf-8"),
        }
    else:
        print(f"[WARN] missing: {p.name}")

print(f"\nLoaded {len(prompts)} prompt templates")

In [ ]:
# ── Prompt-comparison extraction function ─────────────────────────────────────
import re

_FEATURE_BLOCK = "\n".join(
    f'  "{f}": {{"value": "...", "evidence": "...", "confidence": 0.0}}'
    for f in FEATURE_LIST
)
_JSON_SCHEMA = '{\n' + _FEATURE_BLOCK + '\n}'

def extract_with_prompt(ocr_text: str, prompt_template: str, case_id: str) -> dict:
    """
    Substitute OCR text + JSON schema into a prompt template and call Claude.
    Returns a dict of feature_name -> {value, confidence}.
    """
    # Templates use placeholders like {DOCUMENT_TEXT} or just append text
    truncated = ocr_text[:6000]
    for placeholder in ("{DOCUMENT_TEXT}", "[DOCUMENT_TEXT]", "<document>",
                        "INSERT DOCUMENT HERE", "{{document}}"):
        if placeholder in prompt_template:
            user_msg = prompt_template.replace(placeholder, truncated)
            break
    else:
        user_msg = prompt_template + "\n\nDOCUMENT:\n" + truncated

    user_msg += f"\n\nReturn ONLY a JSON object matching this schema:\n{_JSON_SCHEMA}"

    try:
        msg = CLIENT.messages.create(
            model=MODEL, max_tokens=MAX_TOKENS, temperature=0.0,
            messages=[{"role": "user", "content": user_msg}],
        )
        raw = msg.content[0].text
        m   = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            return json.loads(m.group(0))
    except Exception as e:
        return {"error": str(e)}
    return {}

print("Prompt extraction function defined")

In [ ]:
# ── Run comparison — checks cache first ──────────────────────────────────────
cmp_cache = PROMPT_CMP_DIR / "prompt_cmp_results.json"

if cmp_cache.exists():
    cmp_results = json.loads(cmp_cache.read_text(encoding="utf-8"))
    print(f"Loaded {len(cmp_results)} cached comparison results")
else:
    sample_cases = cases[:N_PROMPT_CMP]
    cmp_results  = []

    for case in tqdm(sample_cases, desc="Cases"):
        case_id = case["case_id"]
        ocr     = case["ocr_text"]
        for pname, pdata in tqdm(prompts.items(), desc=f"  {case_id}", leave=False):
            out = extract_with_prompt(ocr, pdata["text"], case_id)
            cmp_results.append({
                "case_id":   case_id,
                "technique": pdata["technique"],
                "prompt_id": pname,
                "features":  out,
            })

    cmp_cache.write_text(json.dumps(cmp_results, indent=2), encoding="utf-8")
    print(f"Saved {len(cmp_results)} comparison results to {cmp_cache}")

In [ ]:
# ── Parse comparison results into flat DataFrame ──────────────────────────────
cmp_rows = []
for r in cmp_results:
    feats = r.get("features", {})
    if "error" in feats:
        continue
    for feat in FEATURE_LIST:
        fd   = feats.get(feat, {})
        val  = fd.get("value", "") if isinstance(fd, dict) else str(fd)
        conf = float(fd.get("confidence", 0.0)) if isinstance(fd, dict) else 0.0
        has_value = (val not in ("", "Not reported", "N/A", "Unknown", "null")
                     and val is not None)
        cmp_rows.append({
            "case_id":    r["case_id"],
            "technique":  r["technique"],
            "prompt_id":  r["prompt_id"],
            "feature":    feat,
            "display":    FEATURES.get(feat, {}).get("display_name", feat),
            "value":      val,
            "confidence": conf,
            "has_value":  has_value,
        })

df_cmp = pd.DataFrame(cmp_rows)
print(df_cmp.shape)
df_cmp.head(3)

In [ ]:
# ── Completeness: % features extracted per technique ─────────────────────────
completeness = (
    df_cmp.groupby(["technique", "prompt_id"])["has_value"]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"has_value": "pct_extracted"})
    .sort_values("pct_extracted", ascending=False)
)

fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.RdYlGn(completeness["pct_extracted"].values / 100)
bars = ax.barh(completeness["prompt_id"], completeness["pct_extracted"],
               color=colors, edgecolor="white")
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=8)
ax.set_xlabel("% features extracted (non-null)")
ax.set_title(f"Extraction completeness by prompt technique  (n={N_PROMPT_CMP} cases)",
             fontsize=12)
ax.set_xlim(0, 110)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "14_prompt_completeness.png", dpi=150)
plt.show()

In [ ]:
# ── Confidence by technique ───────────────────────────────────────────────────
conf_by_tech = (
    df_cmp[df_cmp["has_value"]]
    .groupby("technique")["confidence"]
    .mean()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 4))
conf_by_tech.plot.barh(ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Mean confidence score (0–1)")
ax.set_title("Mean extraction confidence by prompt technique", fontsize=12)
ax.set_xlim(0, 1.05)
for i, v in enumerate(conf_by_tech):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "14_prompt_confidence.png", dpi=150)
plt.show()

In [ ]:
# ── Per-feature completeness heatmap by technique ────────────────────────────
pivot = (
    df_cmp.pivot_table(
        index="prompt_id", columns="display",
        values="has_value", aggfunc="mean"
    ).fillna(0)
)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(pivot * 100, ax=ax, cmap="RdYlGn", vmin=0, vmax=100,
            annot=True, fmt=".0f", linewidths=0.3,
            cbar_kws={"label": "% extracted"})
ax.set_title("Feature extraction completeness (%) — prompt technique × clinical feature",
             fontsize=12)
ax.set_xlabel("")
ax.set_ylabel("Prompt technique")
plt.xticks(rotation=35, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "14_prompt_feature_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 5 · Pipeline Results Summary (RAG Run)

Analysis of the full LangGraph batch run results: verdict distribution, per-feature fabrication/omission rates, domain-level breakdown, and confidence calibration.

In [ ]:
# ── Verdict distribution overview ─────────────────────────────────────────────
verdict_counts = df_features["verdict"].value_counts()
verdict_pct    = verdict_counts / len(df_features) * 100

palette = {"CORRECT": "#2ecc71", "OMISSION": "#f39c12",
           "FABRICATION": "#e74c3c", "UNCERTAIN": "#95a5a6", "": "#bdc3c7"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

verdict_counts.plot.bar(ax=axes[0],
                        color=[palette.get(v, "#ccc") for v in verdict_counts.index],
                        edgecolor="white")
axes[0].set_title("Verdict distribution (count)", fontsize=12)
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

verdict_pct.plot.bar(ax=axes[1],
                     color=[palette.get(v, "#ccc") for v in verdict_pct.index],
                     edgecolor="white")
axes[1].set_title("Verdict distribution (%)", fontsize=12)
axes[1].set_xlabel("")
axes[1].set_ylabel("%")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle(f"Full batch run — {len(all_results)} cases × {len(FEATURE_LIST)} features",
             fontsize=11)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "14_verdict_distribution.png", dpi=150)
plt.show()
print(verdict_counts.to_string())

In [ ]:
# ── Fabrication + Omission rates per feature ──────────────────────────────────
fab_df = (
    df_features.groupby("display")["verdict"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("pct")
    .reset_index()
)

fab_pivot = fab_df.pivot_table(index="display", columns="verdict",
                                values="pct", fill_value=0).reset_index()

feature_order = (
    fab_pivot.get("FABRICATION", pd.Series(0, index=fab_pivot.index))
    .sort_values(ascending=False).index
)
fab_pivot = fab_pivot.loc[feature_order]

fig, ax = plt.subplots(figsize=(13, 6))
x    = np.arange(len(fab_pivot))
w    = 0.28
for i, (col, color, lbl) in enumerate([
    ("CORRECT",     "#2ecc71", "Correct"),
    ("OMISSION",    "#f39c12", "Omission"),
    ("FABRICATION", "#e74c3c", "Fabrication"),
]):
    vals = fab_pivot.get(col, pd.Series(0, index=fab_pivot.index)).fillna(0)
    ax.bar(x + (i - 1) * w, vals, w, label=lbl, color=color, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(fab_pivot["display"].tolist(), rotation=38, ha="right", fontsize=8)
ax.set_ylabel("% of extractions")
ax.set_title("Correct / Omission / Fabrication rate per clinical feature", fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / "14_per_feature_verdict_rates.png", dpi=150)
plt.show()

In [ ]:
# ── Domain-level breakdown (Radiology vs Pathology) ──────────────────────────
RADIOLOGY_FEATURES = {
    "feature_1_lesion_size", "feature_2_lesion_location",
    "feature_3_calcifications_asymmetry", "feature_4_additional_enhancement_mri",
    "feature_5_extent", "feature_6_accurate_clip_placement",
    "feature_7_workup_recommendation", "feature_8_lymph_node",
    "feature_9_chronology_preserved",
}
PATHOLOGY_FEATURES = {
    "feature_10_biopsy_method", "feature_11_invasive_component_size_pathology",
    "feature_12_histologic_diagnosis", "feature_13_receptor_status",
}

df_features["domain"] = df_features["feature"].map(
    lambda f: "Radiology" if f in RADIOLOGY_FEATURES else "Pathology"
)

domain_summary = (
    df_features.groupby(["domain", "verdict"]).size()
    .groupby(level=0).transform(lambda x: x / x.sum() * 100)
    .reset_index(name="pct")
)

fig, ax = plt.subplots(figsize=(8, 4))
domain_pivot = domain_summary.pivot_table(index="domain", columns="verdict",
                                           values="pct", fill_value=0)
domain_pivot[[c for c in ["CORRECT", "OMISSION", "FABRICATION", "UNCERTAIN"]
              if c in domain_pivot.columns]].plot.bar(
    ax=ax, color=["#2ecc71", "#f39c12", "#e74c3c", "#95a5a6"][:len(domain_pivot.columns)],
    edgecolor="white"
)
ax.set_title("Verdict rates by domain (Radiology vs Pathology)", fontsize=12)
ax.set_ylabel("%")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Verdict")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "14_domain_verdict_rates.png", dpi=150)
plt.show()

In [ ]:
# ── Confidence calibration ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
for verdict, color in [("CORRECT","#2ecc71"),("FABRICATION","#e74c3c"),("OMISSION","#f39c12"),("UNCERTAIN","#95a5a6")]:
    sub = df_features[df_features["verdict"] == verdict]["confidence"].dropna()
    if len(sub) > 0:
        ax.hist(sub, bins=20, alpha=0.55, color=color, label=f"{verdict} (n={len(sub)})",
                density=True)
ax.set_xlabel("Extraction confidence score")
ax.set_ylabel("Density")
ax.set_title("Confidence distribution by verdict (RAG pipeline)", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "14_confidence_by_verdict.png", dpi=150)
plt.show()

In [ ]:
# ── Export summary tables ─────────────────────────────────────────────────────
# 1. Per-feature verdict rates
feat_summary = (
    df_features.groupby(["feature", "display", "domain"])["verdict"]
    .value_counts(normalize=True).mul(100).round(2)
    .rename("pct").reset_index()
    .pivot_table(index=["feature", "display", "domain"],
                 columns="verdict", values="pct", fill_value=0)
    .reset_index()
)
feat_summary.to_csv(REPORTS_DIR / "14_feature_verdict_rates.csv", index=False)
print(feat_summary.to_string(index=False))

# 2. Prompt-comparison completeness
completeness.to_csv(REPORTS_DIR / "14_prompt_completeness.csv", index=False)

print(f"\nAll reports saved to {REPORTS_DIR}")

---
## Summary

| Output | Location |
|--------|----------|
| Per-case JSON results | `experiments/runs/{BATCH_RUN_ID}/{case_id}.json` |
| Flat feature table (parquet) | `experiments/runs/{BATCH_RUN_ID}/feature_outputs.parquet` |
| Knowledge graph (GraphML) | `data/knowledge_graph/{BATCH_RUN_ID}_kg.graphml` |
| Prompt comparison results | `experiments/prompt_comparison/prompt_cmp_results.json` |
| All figures | `reports/14_*.png` |
| Feature verdict rates CSV | `reports/14_feature_verdict_rates.csv` |
| Prompt completeness CSV | `reports/14_prompt_completeness.csv` |

**To resume the batch run** (e.g. after a credit/timeout interruption), set `BATCH_RUN_ID` to the same value as the previous run and re-execute Section 2 — already-completed cases are automatically skipped.